# Exploração dos dados do Zenodo (ADA RH Guaíba — Maio/2024)

Verificação do arquivo  (versão v0.4) gerado a partir do Zenodo e copiado para .
O arquivo consolida o impacto observado das inundações de maio de 2024 em 139 municípios da Região Hidrográfica do Guaíba, incluindo população, domicílios, área afetada (total e urbana), vias e estruturas públicas.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

PASTA = Path("..", "..") / "dados" / "processados" / "zenodo"

df = pd.read_csv(PASTA / "analise_ada_rhguaiba_municipios.csv", dtype={"CD_MUN": str})
print(f"Municípios: {len(df)} | Colunas: {len(df.columns)}")
df.head()

## Visão geral e totalizadores da bacia hidrográfica

Total de pessoas e domicílios atingidos na RH do Guaíba durante a enchente de maio de 2024.

In [ ]:
tot_afetados = df["Pessoas_afetadas"].astype(int).sum()
tot_pop_2022 = df["Pessoas_2022"].astype(int).sum()
tot_dom_afetados = df["Domicilios_afetados"].astype(int).sum()
tot_dom_2022 = df["Domicilios_2022"].astype(int).sum()

print(f"População total da bacia (Censo 2022): {tot_pop_2022:,.0f}".replace(",", "."))
print(f"Pessoas diretamente afetadas: {tot_afetados:,.0f} ({tot_afetados/tot_pop_2022:.1%})".replace(",", "."))
print(f"Domicílios diretamente afetados: {tot_dom_afetados:,.0f} ({tot_dom_afetados/tot_dom_2022:.1%})".replace(",", "."))

## Top 15 municípios com mais pessoas afetadas

In [ ]:
top15 = df.sort_values("Pessoas_afetadas", ascending=False).head(15)

fig = px.bar(top15.sort_values("Pessoas_afetadas"),
             x="Pessoas_afetadas",
             y="NM_MUN",
             orientation="h",
             text="Pessoas_afetadas",
             labels={"Pessoas_afetadas": "Pessoas afetadas", "NM_MUN": "Município"},
             title="Top 15 municípios por pessoas diretamente afetadas (RH Guaíba)")
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=500)
fig.show()

**Comentário.** Canoas (154.002 pessoas afetadas, 44,3% da população) e Porto Alegre (151.141 pessoas afetadas, 11,3%) lideram o impacto absoluto, seguidas por São Leopoldo (83.965, 38,6%) e Eldorado do Sul (31.761, 80,3% da cidade).

## Municípios com maior percentual da população atingida

In [ ]:
top_pct = df.sort_values("Pessoas_%", ascending=False).head(15)

fig = px.bar(top_pct.sort_values("Pessoas_%"),
             x="Pessoas_%",
             y="NM_MUN",
             orientation="h",
             text="Pessoas_%",
             labels={"Pessoas_%": "População afetada (%)", "NM_MUN": "Município"},
             title="Top 15 municípios com maior percentual de população afetada")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=500)
fig.show()

## Recorte de Porto Alegre: Comparação com as bases do INCT e IBGE

Verificação da coerência dos totais populacionais e indicadores municipais.

In [ ]:
poa = df[df["CD_MUN"] == "4314902"].iloc[0]

print("=== Porto Alegre (ADA RH Guaíba — Zenodo) ===")
print(f"População 2022: {int(poa["Pessoas_2022"]):,} hab".replace(",", "."))
print(f"Pessoas afetadas: {int(poa["Pessoas_afetadas"]):,} hab ({poa["Pessoas_%"]}%".replace(",", ".") + ")")
print(f"Domicílios afetados: {int(poa["Domicilios_afetados"]):,} ({poa["Domicilios_%"]}%".replace(",", ".") + ")")
print(f"Área urbana atingida: {float(poa["Area_Diretamente_Afetada_Urbana_m2"])/1e6:.2f} km² ({poa["Area_Diretamente_Afetada_Urbana_%"]}%")
print(f"Vias afetadas: {float(poa["Vias_Diretamente_Afetadas_km"]):.1f} km")
print(f"Estruturas afetadas: {int(poa["Estruturas_Afetadas_nro"]):,}".replace(",", "."))
print(f"Equipamentos de saúde atingidos: {poa["Estabelecim_Saude_Afetados_Diretamente"]}")
print(f"Equipamentos de ensino atingidos: {poa["Estabelecim_Ensino_Afetados_Diretamente"]}")
print(f"Estabelecimentos religiosos atingidos: {poa["Estabelecim_Religiosos_Afetados_Diretamente"]}")

**Conclusão.** A população total de 1.332.845 habitantes de Porto Alegre é rigorosamente a mesma encontrada na base de setores censitários do IBGE e na base da simulação do INCT SiMAI. Isso valida o uso conjunto das três bases no projeto, permitindo cruzar os cenários simulados pelo INCT com as perdas reais mapeadas em maio de 2024.